In [1]:
%sql
with expected_ranked as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,

        reporting_culture_organism_key,
        culture_time_shifted,
        culture_key,

        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype

            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as expected_rank

    from workspace.micro_dev.int_reporting_culture_organism

    where culture_time_shifted is not null
),

expected_first as (

    select *
    from expected_ranked
    where expected_rank = 1

),

comparison as (

    select
        e.source_version,
        e.cohort_id,
        e.patient_key,
        e.organism_reporting_group,
        e.reporting_phenotype,

        e.reporting_culture_organism_key
            as expected_reporting_record,

        a.reporting_culture_organism_key
            as actual_reporting_record,

        e.culture_time_shifted
            as expected_time,

        a.culture_time_shifted
            as actual_time

    from expected_first e

    full outer join workspace.micro_dev.int_first_reporting_isolate a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group =
           a.organism_reporting_group
       and e.reporting_phenotype =
           a.reporting_phenotype

)

select
    count(*) as reporting_groups,

    sum(
        case
            when expected_reporting_record
                 = actual_reporting_record
                then 1
            else 0
        end
    ) as correctly_selected,

    sum(
        case
            when expected_reporting_record
                 <> actual_reporting_record
              or expected_reporting_record is null
              or actual_reporting_record is null
                then 1
            else 0
        end
    ) as selection_mismatches,

    case
        when sum(
            case
                when expected_reporting_record
                     <> actual_reporting_record
                  or expected_reporting_record is null
                  or actual_reporting_record is null
                    then 1
                else 0
            end
        ) = 0
            then 'pass'
        else 'fail'
    end as validation_status

from comparison

,reporting_groups,correctly_selected,selection_mismatches,validation_status
0,77369,77369,0,pass


In [2]:
%sql
select
    resolution_status,
    reconciliation_reason,

    count(*) as endpoint_records,

    sum(
        case
            when source_organisms_with_endpoint > 1
                then 1
            else 0
        end
    ) as multi_source_endpoints

from workspace.micro_dev.int_first_reporting_susceptibility

group by
    resolution_status,
    reconciliation_reason

order by
    resolution_status,
    reconciliation_reason

,resolution_status,reconciliation_reason,endpoint_records,multi_source_endpoints
0,conflict,discordant_source_results,278,278
1,conflict,source_endpoint_conflict,3739,118
2,missing,missing_only,967,1
3,resolved,resolved_consistent_result,981699,1911
4,resolved,resolved_with_missing_source_endpoint,2,2


In [3]:
%sql
with key_check as (

    select
        count(*) as row_count,
        count(distinct reporting_susceptibility_key) as unique_key_count
    from workspace.micro_dev.int_first_reporting_susceptibility

),

logic_check as (

    select
        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility is null
                    then 1
                else 0
            end
        ) as resolved_with_null_result,

        sum(
            case
                when resolution_status in ('conflict', 'missing')
                 and susceptibility is not null
                    then 1
                else 0
            end
        ) as unresolved_with_result

    from workspace.micro_dev.int_first_reporting_susceptibility

)

select
    k.row_count,
    k.unique_key_count,

    l.resolved_with_null_result,
    l.unresolved_with_result,

    case
        when k.row_count = k.unique_key_count
         and l.resolved_with_null_result = 0
         and l.unresolved_with_result = 0
            then 'pass'
        else 'fail'
    end as validation_status

from key_check k
cross join logic_check l

,row_count,unique_key_count,resolved_with_null_result,unresolved_with_result,validation_status
0,986685,986685,0,0,pass


In [4]:
%sql
with expected_ranked as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,

        reporting_culture_organism_key,
        culture_time_shifted,
        culture_key,

        row_number() over (
            partition by
                source_version,
                cohort_id,
                patient_key,
                organism_reporting_group,
                reporting_phenotype

            order by
                culture_time_shifted,
                culture_key,
                reporting_culture_organism_key
        ) as expected_rank

    from workspace.micro_dev.int_reporting_culture_organism

    where culture_time_shifted is not null

),

expected_first as (

    select *
    from expected_ranked
    where expected_rank = 1

),

comparison as (

    select
        e.reporting_culture_organism_key
            as expected_reporting_record,

        a.reporting_culture_organism_key
            as actual_reporting_record

    from expected_first e

    full outer join workspace.micro_dev.int_first_reporting_isolate a
        on e.source_version = a.source_version
       and e.cohort_id = a.cohort_id
       and e.patient_key = a.patient_key
       and e.organism_reporting_group = a.organism_reporting_group
       and e.reporting_phenotype = a.reporting_phenotype

)

select
    count(*) as reporting_groups,

    sum(
        case
            when expected_reporting_record = actual_reporting_record
                then 1
            else 0
        end
    ) as correctly_selected,

    sum(
        case
            when expected_reporting_record <> actual_reporting_record
              or expected_reporting_record is null
              or actual_reporting_record is null
                then 1
            else 0
        end
    ) as selection_mismatches,

    case
        when sum(
            case
                when expected_reporting_record <> actual_reporting_record
                  or expected_reporting_record is null
                  or actual_reporting_record is null
                    then 1
                else 0
            end
        ) = 0
            then 'pass'
        else 'fail'
    end as validation_status

from comparison

,reporting_groups,correctly_selected,selection_mismatches,validation_status
0,77369,77369,0,pass


In [5]:
%sql
select
    count(*) as mart_rows,

    count(
        distinct susceptibility_mart_key
    ) as unique_keys,

    sum(
        case
            when eligible_count
                 <> categorized_count
                  + inconclusive_count
                  + synergism_count
                  + missing_result_count
                  + conflict_result_count
                  + untested_count
                then 1
            else 0
        end
    ) as denominator_mismatches,

    sum(
        case
            when reporting_phenotype in (
                'cre',
                'mdr_p_aeruginosa'
            )
                then 1
            else 0
        end
    ) as excluded_mdro_rows

from workspace.micro_dev.mart_susceptibility

,mart_rows,unique_keys,denominator_mismatches,excluded_mdro_rows
0,337,337,0,0
